# Used Car Price Prediction System
**Author:** Arpan Santra  
**Project:** IBM BOB AI/ML Project  
**File:** ArpanSantra_UsedCarPricePrediction.ipynb  

---

## Objective
Build an end-to-end Machine Learning pipeline to predict the resale price of used cars in India using a **Random Forest Regressor**.  
The system is deployed as a **Flask REST API** (backend) and **Streamlit web application** (frontend) — both served from a single `app.py` file.

## Dataset
- **Source:** Used car listings from online platforms across Indian cities
- **File:** `test-data.csv`  
- **Records:** 1,234 rows | **Features:** 13 columns  
- **Cities covered:** Mumbai, Delhi, Bangalore, Chennai, Pune, Hyderabad, Kolkata, Coimbatore, Jaipur, Kochi, Ahmedabad

## Table of Contents
1. Environment Setup & Imports  
2. Data Loading & Exploration  
3. Exploratory Data Analysis (EDA)  
4. Data Preprocessing & Feature Engineering  
5. Model Building & Training  
6. Model Evaluation  
7. Prediction Function  
8. Flask REST API Code  
9. Streamlit Frontend Code  
10. Sample Predictions & Results  

> **Colour Theme:** Blue `#2563eb` &nbsp;·&nbsp; Purple `#7c3aed` &nbsp;·&nbsp; Green `#059669` &nbsp;·&nbsp; Amber `#d97706` &nbsp;·&nbsp; Red `#dc2626`

---
## 1. Environment Setup & Imports

In [ ]:
# ── Install dependencies (run once) ───────────────────────────────────────────
# !pip install flask flask-cors pandas numpy scikit-learn joblib streamlit
#              requests matplotlib seaborn python-docx

import os
import re
import sys
import json
import warnings
import joblib
import numpy  as np
import pandas as pd
import matplotlib.pyplot    as plt
import matplotlib.patches   as mpatches
import seaborn              as sns

from sklearn.ensemble        import RandomForestRegressor
from sklearn.linear_model    import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics         import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline        import Pipeline
from sklearn.compose         import ColumnTransformer
from sklearn.preprocessing   import OrdinalEncoder, StandardScaler
from sklearn.impute          import SimpleImputer

warnings.filterwarnings('ignore')
plt.rcParams['figure.facecolor'] = '#f8f9fa'
plt.rcParams['axes.facecolor']   = '#f8f9fa'
sns.set_theme(style='whitegrid')

print('Python  :', sys.version)
print('Pandas  :', pd.__version__)
print('NumPy   :', np.__version__)
print('Sklearn :', __import__('sklearn').__version__)

---
## 2. Data Loading & Exploration

In [ ]:
# ── Load dataset ───────────────────────────────────────────────────────────────
DATA_PATH = 'test-data.csv'   # adjust path if running from a different directory

df_raw = pd.read_csv(DATA_PATH)
df_raw.drop(columns=[c for c in df_raw.columns if 'Unnamed' in c], inplace=True)

print('Shape :', df_raw.shape)
print('\nColumns:', df_raw.columns.tolist())
df_raw.head()

In [ ]:
# ── Data types and null counts ─────────────────────────────────────────────────
print('=== dtypes ===')
print(df_raw.dtypes)
print('\n=== Null counts ===')
print(df_raw.isnull().sum())

In [ ]:
# ── Descriptive statistics (numeric columns) ───────────────────────────────────
df_raw[['Year', 'Kilometers_Driven', 'Seats']].describe()

In [ ]:
# ── Categorical value distributions ───────────────────────────────────────────
for col in ['Fuel_Type', 'Transmission', 'Owner_Type', 'Location']:
    print(f'\n--- {col} ---')
    print(df_raw[col].value_counts())

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── Global chart style (consistent colour theme across notebook) ──────────────
plt.rcParams.update({
    "figure.facecolor": "#ffffff",
    "axes.facecolor":   "#f8fafc",
    "axes.edgecolor":   "#e2e8f0",
    "axes.labelcolor":  "#64748b",
    "xtick.color":      "#64748b",
    "ytick.color":      "#64748b",
    "text.color":       "#0f172a",
    "grid.color":       "#e2e8f0",
    "grid.linestyle":   "--",
    "grid.linewidth":   0.7,
    "axes.spines.top":  False,
    "axes.spines.right": False,
    "font.size":        10,
})
_PALETTE = ["#2563eb", "#7c3aed", "#059669", "#d97706", "#dc2626"]
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 10, facecolor="#ffffff"))
fig.suptitle('Dataset Distribution Overview', fontsize=16, fontweight='bold')

# 1. Fuel type pie
fuel_counts = df_raw['Fuel_Type'].value_counts()
axes[0][0].pie(fuel_counts.values, labels=fuel_counts.index,
               colors=['#2563eb','#7c3aed','#059669','#d97706'],
               autopct='%1.1f%%', startangle=90,
               wedgeprops={'edgecolor':'white','linewidth':2})
axes[0][0].set_title('Fuel Type Distribution', fontweight='bold', color='#0f172a')

# 2. Transmission bar
tr_counts = df_raw['Transmission'].value_counts()
bars = axes[0][1].bar(tr_counts.index, tr_counts.values,
                       color=['#2563eb','#93c5fd'], width=0.4)
for bar, v in zip(bars, tr_counts.values):
    axes[0][1].text(bar.get_x()+bar.get_width()/2, v+8, str(v),
                    ha='center', fontsize=11)
axes[0][1].set_title('Transmission Distribution', fontweight='bold', color='#0f172a')
axes[0][1].spines[['top','right']].set_visible(False)

# 3. Year histogram
axes[1][0].hist(df_raw['Year'], bins=20, color='#2563eb', edgecolor='white')
axes[1][0].set_title('Manufacturing Year Distribution', fontweight='bold', color='#0f172a')
axes[1][0].set_xlabel('Year')
axes[1][0].set_ylabel('Count')
axes[1][0].spines[['top','right']].set_visible(False)

# 4. Location bar
loc_counts = df_raw['Location'].value_counts().sort_values(ascending=True)
axes[1][1].barh(loc_counts.index, loc_counts.values, color='#2563eb', alpha=0.85)
axes[1][1].set_title('Cars per Location', fontweight='bold', color='#0f172a')
axes[1][1].spines[['top','right']].set_visible(False)
axes[1][1].set_xlabel('Count')

plt.tight_layout(pad=1.3)
plt.show()

In [ ]:
# ── Global chart style (consistent colour theme across notebook) ──────────────
plt.rcParams.update({
    "figure.facecolor": "#ffffff",
    "axes.facecolor":   "#f8fafc",
    "axes.edgecolor":   "#e2e8f0",
    "axes.labelcolor":  "#64748b",
    "xtick.color":      "#64748b",
    "ytick.color":      "#64748b",
    "text.color":       "#0f172a",
    "grid.color":       "#e2e8f0",
    "grid.linestyle":   "--",
    "grid.linewidth":   0.7,
    "axes.spines.top":  False,
    "axes.spines.right": False,
    "font.size":        10,
})
_PALETTE = ["#2563eb", "#7c3aed", "#059669", "#d97706", "#dc2626"]
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5, facecolor="#ffffff"))
fig.suptitle('Kilometers Driven & Owner Type Analysis', fontsize=14, fontweight='bold')

# KM driven distribution
axes[0].hist(df_raw['Kilometers_Driven'], bins=40,
             color='#7c3aed', edgecolor='white')
axes[0].set_title('Kilometers Driven Distribution', fontweight='bold', color='#0f172a')
axes[0].set_xlabel('Kilometers Driven')
axes[0].set_ylabel('Frequency')
axes[0].spines[['top','right']].set_visible(False)

# Owner type
ow_counts = df_raw['Owner_Type'].value_counts()
bars2 = axes[1].bar(ow_counts.index, ow_counts.values,
                    color=['#2563eb','#7c3aed','#059669','#d97706'])
for bar, v in zip(bars2, ow_counts.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+5, str(v),
                 ha='center', fontsize=10)
axes[1].set_title('Owner Type Distribution', fontweight='bold', color='#0f172a')
axes[1].spines[['top','right']].set_visible(False)
plt.xticks(rotation=15, ha='right')

plt.tight_layout(pad=1.3)
plt.show()

In [ ]:
# New_Price availability analysis
has_price = df_raw['New_Price'].notna().sum()
no_price  = df_raw['New_Price'].isna().sum()
print(f'New_Price available : {has_price} rows ({100*has_price/len(df_raw):.1f}%)')
print(f'New_Price missing   : {no_price} rows ({100*no_price/len(df_raw):.1f}%)')

# Sample of non-null New_Price
df_raw[df_raw['New_Price'].notna()][['Name','Year','Fuel_Type','New_Price']].head(8)

---
## 4. Data Preprocessing & Feature Engineering

In [ ]:
# ── Helper functions ───────────────────────────────────────────────────────────

def extract_numeric(series: pd.Series) -> pd.Series:
    """Pull the first number from a string column e.g. '998 CC' -> 998.0"""
    return pd.to_numeric(
        series.astype(str).str.extract(r'([\d.]+)', expand=False),
        errors='coerce'
    )

def extract_mileage(series: pd.Series) -> pd.Series:
    """Normalise mileage; convert km/kg to kmpl (*1.4)."""
    def _parse(val):
        if pd.isna(val):
            return np.nan
        val = str(val).strip()
        m = re.match(r'([\d.]+)\s*(kmpl|km/kg)?', val, re.I)
        if not m:
            return np.nan
        num  = float(m.group(1))
        unit = (m.group(2) or '').strip().lower()
        return num * 1.4 if unit == 'km/kg' else num
    return series.apply(_parse)

def extract_price(series: pd.Series) -> pd.Series:
    """Convert 'XX.XX Lakh' -> float"""
    return pd.to_numeric(
        series.astype(str).str.extract(r'([\d.]+)', expand=False),
        errors='coerce'
    )

print('Helper functions defined.')

In [ ]:
# ── Full preprocessing pipeline ────────────────────────────────────────────────

def load_and_preprocess(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.drop(columns=[c for c in df.columns if 'Unnamed' in c], inplace=True)

    # Feature: brand from name
    df['Brand'] = df['Name'].str.split().str[0]

    # Numeric conversions
    df['Mileage']   = extract_mileage(df['Mileage'])
    df['Engine']    = extract_numeric(df['Engine'])
    df['Power']     = extract_numeric(df['Power'])
    df['New_Price'] = extract_price(df['New_Price'])

    # Derived feature: car age
    df['Car_Age'] = 2024 - df['Year']

    # ── Build target variable (Price in Lakh) ──────────────────────────────────
    # Use New_Price * depreciation where available; impute rest via brand median
    def depreciation(row):
        age = max(1, 2024 - row['Year'])
        km  = row['Kilometers_Driven']
        return max(0.1, 1 - 0.07*age) * max(0.5, 1 - km/700000)

    df['_dep']  = df.apply(depreciation, axis=1)
    df['Price'] = df['New_Price'] * df['_dep']

    has_price    = df['Price'].notna()
    brand_median = df[has_price].groupby('Brand')['Price'].median()
    overall_med  = df[has_price]['Price'].median()

    def fill_price(row):
        if pd.notna(row['Price']):
            return row['Price']
        bmed = brand_median.get(row['Brand'], overall_med)
        return bmed * row['_dep']

    df['Price'] = df.apply(fill_price, axis=1)
    df.drop(columns=['_dep', 'New_Price'], inplace=True)
    df.dropna(subset=['Price'], inplace=True)

    # Clip extreme prices (top 1%)
    q99 = df['Price'].quantile(0.99)
    df  = df[df['Price'] <= q99].copy()

    return df

df = load_and_preprocess(DATA_PATH)
print('Preprocessed shape:', df.shape)
df.head()

In [ ]:
# ── Feature overview after preprocessing ──────────────────────────────────────
print('Null counts after preprocessing:')
print(df.isnull().sum())
print('\nPrice stats (Lakh INR):')
print(df['Price'].describe())

In [ ]:
# ── Global chart style (consistent colour theme across notebook) ──────────────
plt.rcParams.update({
    "figure.facecolor": "#ffffff",
    "axes.facecolor":   "#f8fafc",
    "axes.edgecolor":   "#e2e8f0",
    "axes.labelcolor":  "#64748b",
    "xtick.color":      "#64748b",
    "ytick.color":      "#64748b",
    "text.color":       "#0f172a",
    "grid.color":       "#e2e8f0",
    "grid.linestyle":   "--",
    "grid.linewidth":   0.7,
    "axes.spines.top":  False,
    "axes.spines.right": False,
    "font.size":        10,
})
_PALETTE = ["#2563eb", "#7c3aed", "#059669", "#d97706", "#dc2626"]
# ─────────────────────────────────────────────────────────────────────────────

# ── Target variable distribution ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4, facecolor="#ffffff"))

axes[0].hist(df['Price'], bins=40, color='#2563eb', edgecolor='white')
axes[0].set_title('Price Distribution (Lakh INR)', fontweight='bold')
axes[0].set_xlabel('Price (Lakh)')
axes[0].set_ylabel('Frequency')
axes[0].spines[['top','right']].set_visible(False)

# Price by fuel type
fuel_price = df.groupby('Fuel_Type')['Price'].median().sort_values(ascending=False)
bars3 = axes[1].bar(fuel_price.index, fuel_price.values,
                    color=['#2563eb','#7c3aed','#059669','#d97706'])
for bar, v in zip(bars3, fuel_price.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+0.1,
                 f'{v:.2f}L', ha='center', fontsize=10, fontweight='bold')
axes[1].set_title('Median Price by Fuel Type', fontweight='bold', color='#0f172a')
axes[1].set_ylabel('Median Price (Lakh)')
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout(pad=1.3)
plt.show()

In [ ]:
# ── Global chart style (consistent colour theme across notebook) ──────────────
plt.rcParams.update({
    "figure.facecolor": "#ffffff",
    "axes.facecolor":   "#f8fafc",
    "axes.edgecolor":   "#e2e8f0",
    "axes.labelcolor":  "#64748b",
    "xtick.color":      "#64748b",
    "ytick.color":      "#64748b",
    "text.color":       "#0f172a",
    "grid.color":       "#e2e8f0",
    "grid.linestyle":   "--",
    "grid.linewidth":   0.7,
    "axes.spines.top":  False,
    "axes.spines.right": False,
    "font.size":        10,
})
_PALETTE = ["#2563eb", "#7c3aed", "#059669", "#d97706", "#dc2626"]
# ─────────────────────────────────────────────────────────────────────────────

# ── Correlation heatmap ────────────────────────────────────────────────────────
num_cols = ['Car_Age', 'Kilometers_Driven', 'Mileage', 'Engine', 'Power', 'Seats', 'Price']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7, facecolor="#ffffff"))
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, ax=ax, linewidths=0.5,
            annot_kws={'size':10})
ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold', color='#0f172a', pad=12)
plt.tight_layout(pad=1.3)
plt.show()

In [ ]:
# ── Global chart style (consistent colour theme across notebook) ──────────────
plt.rcParams.update({
    "figure.facecolor": "#ffffff",
    "axes.facecolor":   "#f8fafc",
    "axes.edgecolor":   "#e2e8f0",
    "axes.labelcolor":  "#64748b",
    "xtick.color":      "#64748b",
    "ytick.color":      "#64748b",
    "text.color":       "#0f172a",
    "grid.color":       "#e2e8f0",
    "grid.linestyle":   "--",
    "grid.linewidth":   0.7,
    "axes.spines.top":  False,
    "axes.spines.right": False,
    "font.size":        10,
})
_PALETTE = ["#2563eb", "#7c3aed", "#059669", "#d97706", "#dc2626"]
# ─────────────────────────────────────────────────────────────────────────────

# ── Power vs Price scatter ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4, facecolor="#ffffff"))

axes[0].scatter(df['Power'], df['Price'], alpha=0.4, color='#2563eb', s=20)
axes[0].set_xlabel('Power (bhp)', fontsize=11)
axes[0].set_ylabel('Price (Lakh)', fontsize=11)
axes[0].set_title('Power vs Price', fontweight='bold', color='#0f172a')
axes[0].spines[['top','right']].set_visible(False)

axes[1].scatter(df['Car_Age'], df['Price'], alpha=0.4, color='#7c3aed', s=20)
axes[1].set_xlabel('Car Age (years)', fontsize=11)
axes[1].set_ylabel('Price (Lakh)', fontsize=11)
axes[1].set_title('Car Age vs Price', fontweight='bold', color='#0f172a')
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout(pad=1.3)
plt.show()

---
## 5. Model Building & Training

In [ ]:
# ── Define features and target ─────────────────────────────────────────────────
CATEGORICAL_FEATURES = ['Brand', 'Location', 'Fuel_Type', 'Transmission', 'Owner_Type']
NUMERIC_FEATURES     = ['Car_Age', 'Kilometers_Driven', 'Mileage', 'Engine', 'Power', 'Seats']
TARGET               = 'Price'

X = df[CATEGORICAL_FEATURES + NUMERIC_FEATURES]
y = df[TARGET]

print('Feature matrix shape:', X.shape)
print('Target series shape :', y.shape)
print('\nCategorical features:', CATEGORICAL_FEATURES)
print('Numeric features    :', NUMERIC_FEATURES)

In [ ]:
# ── Train / Test split ─────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f'Train rows : {len(X_train)}  ({100*len(X_train)/len(X):.0f}%)')
print(f'Test rows  : {len(X_test)}   ({100*len(X_test)/len(X):.0f}%)')

In [ ]:
# ── sklearn Pipeline ──────────────────────────────────────────────────────────
def build_pipeline() -> Pipeline:
    numeric_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
    ])
    categorical_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ])
    preprocessor = ColumnTransformer([
        ('num', numeric_transformer,     NUMERIC_FEATURES),
        ('cat', categorical_transformer, CATEGORICAL_FEATURES),
    ])
    return Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', RandomForestRegressor(
            n_estimators=200,
            max_depth=15,
            min_samples_split=4,
            random_state=42,
            n_jobs=-1
        )),
    ])

print('Pipeline structure:')
print(build_pipeline())

In [ ]:
# ── Train the model ────────────────────────────────────────────────────────────
print('Training Random Forest Regressor ...')
rf_pipeline = build_pipeline()
rf_pipeline.fit(X_train, y_train)
print('Training complete.')

---
## 6. Model Evaluation

In [ ]:
# ── Test-set metrics ───────────────────────────────────────────────────────────
y_pred = rf_pipeline.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print('=' * 40)
print('       MODEL EVALUATION RESULTS')
print('=' * 40)
print(f'  MAE        : {mae:.4f}  Lakh INR')
print(f'  RMSE       : {rmse:.4f}  Lakh INR')
print(f'  R2 Score   : {r2:.4f}')
print('=' * 40)

In [ ]:
# ── 5-Fold Cross-Validation ────────────────────────────────────────────────────
print('Running 5-fold cross-validation ...')
cv_scores = cross_val_score(build_pipeline(), X, y, cv=5, scoring='r2', n_jobs=-1)
print(f'CV R2 scores : {cv_scores.round(4)}')
print(f'Mean         : {cv_scores.mean():.4f}')
print(f'Std Dev      : {cv_scores.std():.4f}')

In [ ]:
# ── Global chart style (consistent colour theme across notebook) ──────────────
plt.rcParams.update({
    "figure.facecolor": "#ffffff",
    "axes.facecolor":   "#f8fafc",
    "axes.edgecolor":   "#e2e8f0",
    "axes.labelcolor":  "#64748b",
    "xtick.color":      "#64748b",
    "ytick.color":      "#64748b",
    "text.color":       "#0f172a",
    "grid.color":       "#e2e8f0",
    "grid.linestyle":   "--",
    "grid.linewidth":   0.7,
    "axes.spines.top":  False,
    "axes.spines.right": False,
    "font.size":        10,
})
_PALETTE = ["#2563eb", "#7c3aed", "#059669", "#d97706", "#dc2626"]
# ─────────────────────────────────────────────────────────────────────────────

# ── Residual analysis plots ────────────────────────────────────────────────────
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 3, figsize=(16, 5, facecolor="#ffffff"))
fig.suptitle('Model Residual Analysis', fontsize=14, fontweight='bold')

# Actual vs Predicted
axes[0].scatter(y_test, y_pred, alpha=0.4, color='#2563eb', s=20)
mn, mx = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
axes[0].plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect fit')
axes[0].set_xlabel('Actual Price (Lakh)')
axes[0].set_ylabel('Predicted Price (Lakh)')
axes[0].set_title('Actual vs Predicted')
axes[0].legend()
axes[0].spines[['top','right']].set_visible(False)

# Residual distribution
axes[1].hist(residuals, bins=40, color='#7c3aed', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual (Lakh)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution')
axes[1].spines[['top','right']].set_visible(False)

# Residuals vs Predicted
axes[2].scatter(y_pred, residuals, alpha=0.4, color='#059669', s=20)
axes[2].axhline(0, color='red', linestyle='--')
axes[2].set_xlabel('Predicted Price (Lakh)')
axes[2].set_ylabel('Residual')
axes[2].set_title('Residuals vs Predicted')
axes[2].spines[['top','right']].set_visible(False)

plt.tight_layout(pad=1.3)
plt.show()

In [ ]:
# ── Global chart style (consistent colour theme across notebook) ──────────────
plt.rcParams.update({
    "figure.facecolor": "#ffffff",
    "axes.facecolor":   "#f8fafc",
    "axes.edgecolor":   "#e2e8f0",
    "axes.labelcolor":  "#64748b",
    "xtick.color":      "#64748b",
    "ytick.color":      "#64748b",
    "text.color":       "#0f172a",
    "grid.color":       "#e2e8f0",
    "grid.linestyle":   "--",
    "grid.linewidth":   0.7,
    "axes.spines.top":  False,
    "axes.spines.right": False,
    "font.size":        10,
})
_PALETTE = ["#2563eb", "#7c3aed", "#059669", "#d97706", "#dc2626"]
# ─────────────────────────────────────────────────────────────────────────────

# ── Feature importance ─────────────────────────────────────────────────────────
rf_model   = rf_pipeline.named_steps['regressor']
feat_names = NUMERIC_FEATURES + CATEGORICAL_FEATURES
importances = rf_model.feature_importances_

feat_df = pd.DataFrame({'Feature': feat_names, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5, facecolor="#ffffff"))
bars = ax.barh(feat_df['Feature'], feat_df['Importance'],
               color='#2563eb', alpha=0.85)
ax.set_title('Feature Importances — Random Forest', fontsize=12, fontweight='bold', color='#0f172a')
ax.set_xlabel('Importance Score')
ax.spines[['top','right']].set_visible(False)
for bar, v in zip(bars, feat_df['Importance']):
    ax.text(v+0.002, bar.get_y()+bar.get_height()/2,
            f'{v:.3f}', va='center', fontsize=9)
plt.tight_layout(pad=1.3)
plt.show()

In [ ]:
# ── Global chart style (consistent colour theme across notebook) ──────────────
plt.rcParams.update({
    "figure.facecolor": "#ffffff",
    "axes.facecolor":   "#f8fafc",
    "axes.edgecolor":   "#e2e8f0",
    "axes.labelcolor":  "#64748b",
    "xtick.color":      "#64748b",
    "ytick.color":      "#64748b",
    "text.color":       "#0f172a",
    "grid.color":       "#e2e8f0",
    "grid.linestyle":   "--",
    "grid.linewidth":   0.7,
    "axes.spines.top":  False,
    "axes.spines.right": False,
    "font.size":        10,
})
_PALETTE = ["#2563eb", "#7c3aed", "#059669", "#d97706", "#dc2626"]
# ─────────────────────────────────────────────────────────────────────────────

# ── Metrics summary bar chart ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4, facecolor="#ffffff"))
m_names  = ['R2 Score', 'MAE (Lakh)', 'RMSE (Lakh)', 'CV R2']
m_values = [round(r2,4), round(mae,4), round(rmse,4), round(cv_scores.mean(),4)]
colors   = ['#2563eb','#7c3aed','#059669','#d97706']
bars4 = ax.bar(m_names, m_values, color=colors, width=0.5)
for bar, v in zip(bars4, m_values):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.02,
            str(v), ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_title('Model Performance Metrics', fontsize=13, fontweight='bold', color='#0f172a')
ax.set_ylim(0, max(m_values)*1.3)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(pad=1.3)
plt.show()

print('\nSummary:')
for n, v in zip(m_names, m_values):
    print(f'  {n:<15}: {v}')

In [ ]:
# ── Save trained model & metadata ─────────────────────────────────────────────
MODEL_DIR = 'models'
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(rf_pipeline, os.path.join(MODEL_DIR, 'used_car_rf_model.pkl'))

meta = {
    'features'     : CATEGORICAL_FEATURES + NUMERIC_FEATURES,
    'categorical'  : CATEGORICAL_FEATURES,
    'numeric'      : NUMERIC_FEATURES,
    'target'       : TARGET,
    'mae'          : round(mae, 4),
    'rmse'         : round(rmse, 4),
    'r2'           : round(r2, 4),
    'cv_r2_mean'   : round(cv_scores.mean(), 4),
    'cv_r2_std'    : round(cv_scores.std(), 4),
    'train_rows'   : int(len(X_train)),
    'test_rows'    : int(len(X_test)),
    'total_rows'   : int(len(df)),
    'brands'       : sorted(df['Brand'].unique().tolist()),
    'locations'    : sorted(df['Location'].unique().tolist()),
    'fuel_types'   : sorted(df['Fuel_Type'].unique().tolist()),
    'transmissions': sorted(df['Transmission'].unique().tolist()),
    'owner_types'  : sorted(df['Owner_Type'].unique().tolist()),
    'year_min'     : int(df['Year'].min()),
    'year_max'     : int(df['Year'].max()),
    'km_min'       : int(df['Kilometers_Driven'].min()),
    'km_max'       : int(df['Kilometers_Driven'].max()),
}
joblib.dump(meta, os.path.join(MODEL_DIR, 'model_meta.pkl'))
print('Model saved  ->', os.path.join(MODEL_DIR, 'used_car_rf_model.pkl'))
print('Metadata saved ->', os.path.join(MODEL_DIR, 'model_meta.pkl'))

---
## 7. Prediction Function

In [ ]:
# ── Single-car prediction function ────────────────────────────────────────────
def predict_price(brand, location, year, km_driven, fuel_type,
                  transmission, owner_type, mileage_kmpl,
                  engine_cc, power_bhp, seats):
    """
    Predicts the resale price of a used car.
    Returns dict with predicted price and confidence range.
    """
    row = pd.DataFrame([{
        'Brand'            : brand,
        'Location'         : location,
        'Fuel_Type'        : fuel_type,
        'Transmission'     : transmission,
        'Owner_Type'       : owner_type,
        'Car_Age'          : 2024 - int(year),
        'Kilometers_Driven': int(km_driven),
        'Mileage'          : float(mileage_kmpl),
        'Engine'           : float(engine_cc),
        'Power'            : float(power_bhp),
        'Seats'            : float(seats),
    }])
    price = round(max(0.1, float(rf_pipeline.predict(row)[0])), 2)
    return {
        'predicted_price': price,
        'unit'           : 'Lakh INR',
        'price_range'    : {'low': round(price*0.90, 2), 'high': round(price*1.10, 2)}
    }

print('predict_price() function ready.')

---
## 8. Flask REST API Code

> **Note:** The Flask server cannot run inside a Jupyter notebook cell directly.  
> Save the code below as `backend/app.py` and run with `python app.py`.
> The same `app.py` file also contains the Streamlit frontend (Section 9).
> Run the UI with `streamlit run app.py`.

In [ ]:
FLASK_CODE = '''
"""
backend/app.py  -  ONE file for Flask API + Streamlit UI
  python app.py          -> Flask REST API  (port 5000)
  python -m streamlit run app.py -> Streamlit UI    (port 8501)
"""

import os, re, sys
import numpy as np
import pandas as pd
import joblib

_IS_STREAMLIT = "streamlit" in sys.modules or any("streamlit" in a for a in sys.argv)

BASE_DIR   = os.path.dirname(os.path.abspath(__file__))
MODEL_PATH = os.path.join(BASE_DIR, "..", "models", "used_car_rf_model.pkl")
META_PATH  = os.path.join(BASE_DIR, "..", "models", "model_meta.pkl")
DATA_PATH  = os.path.join(BASE_DIR, "..", "test-data.csv")

# --- Shared helpers ----------------------------------------------------------
def _extract_numeric(val):
    if val is None or (isinstance(val, float) and np.isnan(val)): return None
    m = re.search(r"([\\d.]+)", str(val))
    return float(m.group(1)) if m else None

def _extract_mileage(val):
    if val is None or (isinstance(val, float) and np.isnan(val)): return None
    val = str(val).strip()
    m = re.match(r"([\\d.]+)\\s*(kmpl|km/kg)?", val, re.I)
    if not m: return None
    num = float(m.group(1))
    return num * 1.4 if (m.group(2) or "").strip().lower() == "km/kg" else num

def build_input_df(data):
    year = int(data.get("year", 2018))
    return pd.DataFrame([{
        "Brand": data.get("brand","Maruti"), "Location": data.get("location","Mumbai"),
        "Fuel_Type": data.get("fuel_type","Petrol"), "Transmission": data.get("transmission","Manual"),
        "Owner_Type": data.get("owner_type","First"), "Car_Age": 2024 - year,
        "Kilometers_Driven": int(data.get("kilometers_driven",50000)),
        "Mileage": _extract_mileage(str(data.get("mileage","18 kmpl"))),
        "Engine":  _extract_numeric(str(data.get("engine","1200 CC"))),
        "Power":   _extract_numeric(str(data.get("power","80 bhp"))),
        "Seats":   float(data.get("seats",5)),
    }])

model, meta = None, {}
try:
    model = joblib.load(MODEL_PATH)
    meta  = joblib.load(META_PATH)
except Exception as e:
    print(f"[WARN] {e}")

# --- Flask section -----------------------------------------------------------
if not _IS_STREAMLIT:
    from flask import Flask, request, jsonify
    from flask_cors import CORS
    app = Flask(__name__)
    CORS(app)

    @app.route("/api/health")   # GET
    def health(): return jsonify({"status":"ok","model_loaded":model is not None})

    @app.route("/api/predict", methods=["POST"])
    def predict():
        data  = request.get_json(force=True)
        price = round(max(0.1, float(model.predict(build_input_df(data))[0])), 2)
        return jsonify({"predicted_price": price, "unit": "Lakh INR",
                        "price_range": {"low": round(price*0.9,2), "high": round(price*1.1,2)}})

    if __name__ == "__main__":
        app.run(host="127.0.0.1", port=5000, debug=False)
'''
print('Flask API code (save to backend/app.py):')
print(FLASK_CODE)

---
## 9. Streamlit Frontend Code

> The Streamlit UI is embedded in the same `backend/app.py` file.  
> Launch it with: `streamlit run backend/app.py`

In [ ]:
STREAMLIT_SNIPPET = '''
# Streamlit section (inside backend/app.py, executed when running via streamlit)
else:
    import streamlit as st
    import requests, matplotlib.pyplot as plt

    st.set_page_config(page_title="Used Car Price Predictor", page_icon="car", layout="wide")

    # Sidebar inputs
    with st.sidebar:
        brand        = st.selectbox("Brand",    meta.get("brands", ["Maruti"]))
        location     = st.selectbox("Location", meta.get("locations", ["Mumbai"]))
        year         = st.slider("Year", 1996, 2019, 2016)
        km_driven    = st.number_input("KM Driven", 1000, 350000, 45000)
        fuel_type    = st.selectbox("Fuel",  meta.get("fuel_types", ["Petrol"]))
        transmission = st.selectbox("Gearbox", meta.get("transmissions", ["Manual"]))
        owner_type   = st.selectbox("Owner",  meta.get("owner_types", ["First"]))
        mileage      = st.number_input("Mileage (kmpl)", 5.0, 35.0, 18.0)
        engine       = st.number_input("Engine (CC)",   600, 5000, 1200)
        power        = st.number_input("Power (bhp)",  30.0, 600.0, 80.0)
        seats        = st.selectbox("Seats", [2,4,5,6,7,8])
        btn          = st.button("Predict Price", type="primary")

    if btn:
        payload = {"brand": brand, "location": location, "year": year,
                   "kilometers_driven": km_driven, "fuel_type": fuel_type,
                   "transmission": transmission, "owner_type": owner_type,
                   "mileage": f"{mileage} kmpl", "engine": f"{engine} CC",
                   "power": f"{power} bhp", "seats": seats}
        res   = requests.post("http://127.0.0.1:5000/api/predict", json=payload).json()
        price = res["predicted_price"]
        st.success(f"Predicted Price: Rs. {price} Lakh")
'''
print('Streamlit frontend snippet (part of backend/app.py):')
print(STREAMLIT_SNIPPET)

---
## 10. Sample Predictions & Results

In [ ]:
# ── Run sample predictions ─────────────────────────────────────────────────────
test_cases = [
    dict(brand='Maruti',  location='Mumbai',    year=2017, km_driven=40000,
         fuel_type='Petrol',  transmission='Manual',    owner_type='First',
         mileage_kmpl=20, engine_cc=1200, power_bhp=82,  seats=5),
    dict(brand='Toyota',  location='Delhi',     year=2015, km_driven=80000,
         fuel_type='Diesel',  transmission='Automatic', owner_type='Second',
         mileage_kmpl=16, engine_cc=2400, power_bhp=148, seats=7),
    dict(brand='Hyundai', location='Bangalore', year=2019, km_driven=15000,
         fuel_type='Petrol',  transmission='Manual',    owner_type='First',
         mileage_kmpl=22, engine_cc=1000, power_bhp=67,  seats=5),
    dict(brand='Honda',   location='Chennai',   year=2016, km_driven=60000,
         fuel_type='Petrol',  transmission='Manual',    owner_type='Second',
         mileage_kmpl=18, engine_cc=1500, power_bhp=120, seats=5),
    dict(brand='BMW',     location='Mumbai',    year=2018, km_driven=30000,
         fuel_type='Diesel',  transmission='Automatic', owner_type='First',
         mileage_kmpl=14, engine_cc=1995, power_bhp=190, seats=5),
]

results = []
print(f'{"#":<3} {"Brand":<10} {"Year":<6} {"Fuel":<8} {"KM":>8}   {"Predicted":>12}  {"Range"}')
print('-' * 72)
for i, tc in enumerate(test_cases, 1):
    res   = predict_price(**tc)
    price = res['predicted_price']
    lo    = res['price_range']['low']
    hi    = res['price_range']['high']
    results.append({'car': f"{tc['brand']} ({tc['year']})",
                    'price': price, 'low': lo, 'high': hi})
    print(f"{i:<3} {tc['brand']:<10} {tc['year']:<6} {tc['fuel_type']:<8} "
          f"{tc['km_driven']:>8,}   Rs. {price:>6.2f} L   "
          f"[{lo:.2f} - {hi:.2f}]")

In [ ]:
# ── Global chart style (consistent colour theme across notebook) ──────────────
plt.rcParams.update({
    "figure.facecolor": "#ffffff",
    "axes.facecolor":   "#f8fafc",
    "axes.edgecolor":   "#e2e8f0",
    "axes.labelcolor":  "#64748b",
    "xtick.color":      "#64748b",
    "ytick.color":      "#64748b",
    "text.color":       "#0f172a",
    "grid.color":       "#e2e8f0",
    "grid.linestyle":   "--",
    "grid.linewidth":   0.7,
    "axes.spines.top":  False,
    "axes.spines.right": False,
    "font.size":        10,
})
_PALETTE = ["#2563eb", "#7c3aed", "#059669", "#d97706", "#dc2626"]
# ─────────────────────────────────────────────────────────────────────────────

# ── Visualise sample predictions ───────────────────────────────────────────────
import matplotlib.ticker as mtick

cars   = [r['car']   for r in results]
prices = [r['price'] for r in results]
lows   = [r['low']   for r in results]
highs  = [r['high']  for r in results]

x = np.arange(len(cars))
fig, ax = plt.subplots(figsize=(12, 5, facecolor="#ffffff"))

bars = ax.bar(x, prices, width=0.5,
              color=['#2563eb','#7c3aed','#059669','#d97706','#dc2626'],
              zorder=3, label='Predicted Price')

for xi, (lo, hi, p) in enumerate(zip(lows, highs, prices)):
    ax.plot([xi, xi], [lo, hi], color='#94a3b8', lw=3, zorder=2)
    ax.scatter([xi, xi], [lo, hi], color='#94a3b8', s=80, zorder=4)

for bar, v in zip(bars, prices):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.1,
            f'Rs.{v:.2f}L', ha='center', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(cars, fontsize=9)
ax.set_ylabel('Predicted Price (Lakh INR)', fontsize=11)
ax.set_title('Sample Predictions — Used Car Price Estimator', fontsize=13, fontweight='bold', color='#0f172a')
ax.spines[['top','right']].set_visible(False)
ax.set_ylim(0, max(highs)*1.3)

price_patch = mpatches.Patch(color='#2563eb', label='Predicted Price')
range_patch = mpatches.Patch(color='#94a3b8', label='Price Range (+/-10%)')
ax.legend(handles=[price_patch, range_patch], fontsize=10)

plt.tight_layout(pad=1.3)
plt.show()

In [ ]:
# ── Final project summary ──────────────────────────────────────────────────────
print('=' * 55)
print('   USED CAR PRICE PREDICTION — PROJECT SUMMARY')
print('=' * 55)
print(f'  Author        : Arpan Santra')
print(f'  Algorithm     : Random Forest Regressor')
print(f'  Dataset rows  : {len(df)}')
print(f'  Features used : {len(CATEGORICAL_FEATURES + NUMERIC_FEATURES)}')
print(f'  Train / Test  : {len(X_train)} / {len(X_test)}')
print('-' * 55)
print(f'  R2 Score      : {r2:.4f}')
print(f'  MAE           : {mae:.4f} Lakh INR')
print(f'  RMSE          : {rmse:.4f} Lakh INR')
print(f'  CV R2 (5-fold): {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')
print('-' * 55)
print('  Backend  : Flask REST API       -> python backend/app.py')
print('  Frontend : Streamlit Web App    -> streamlit run backend/app.py')
print('  API URL  : http://127.0.0.1:5000')
print('  UI  URL  : http://localhost:8501')
print('=' * 55)